# LTX-Video 2.3 OpenVINO conversion notebook

This notebook shows how to load **`dg845/LTX-2.3-Diffusers`**, convert the main LTX-2.3 components to OpenVINO IR, run CPU/GPU smoke tests, and save a short reference video.

> Notes
> - LTX-2.3 uses **Gemma hidden states → text connectors → transformer**.
> - The transformer expects **connector outputs** (`4096` video channels / `2048` audio channels), not raw Gemma hidden states.
> - The end-to-end generation example uses the original Diffusers pipeline as a reference output, while the OpenVINO section validates converted components individually.

In [ ]:
%pip install -q --upgrade openvino diffusers transformers accelerate sentencepiece safetensors huggingface_hub imageio imageio-ffmpeg ipywidgets

In [ ]:
from pathlib import Path
import json
import numpy as np
import openvino as ov
import torch
import torch.nn as nn
import imageio.v3 as iio

from diffusers import LTX2Pipeline
from diffusers.models.transformers.transformer_ltx2 import LTX2VideoTransformer3DModel

MODEL_ID = 'dg845/LTX-2.3-Diffusers'
OUT_DIR = Path('ov_model')
OUT_DIR.mkdir(parents=True, exist_ok=True)

core = ov.Core()
print('OpenVINO:', ov.__version__)

In [ ]:
pipe = LTX2Pipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
    pipe.transformer.eval()
    pipe.text_encoder.eval()
    pipe.vae.eval()

    print('Transformer config:')
    print(json.dumps(pipe.transformer.config.to_dict(), indent=2)[:1200])
    print('
Connectors config:')
    print(json.dumps(pipe.connectors.config.to_dict(), indent=2)[:1200])

## Conversion helpers

In [ ]:
class TextEncoderWrapper(nn.Module):
    def __init__(self, text_encoder):
        super().__init__()
        self.text_encoder = text_encoder.eval()

    def forward(self, input_ids, attention_mask):
        outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        hidden_states = torch.stack(outputs.hidden_states, dim=-1)
        return hidden_states.flatten(2, 3)


class VAEEncoderWrapper(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae.eval()

    def forward(self, video):
        return self.vae.encode(video).latent_dist.sample()


class VAEDecoderWrapper(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae.eval()

    def forward(self, latents):
        return self.vae.decode(latents).sample


class TransformerWrapper(nn.Module):
    def __init__(self, transformer):
        super().__init__()
        self.transformer = transformer.eval()

    def forward(
        self,
        hidden_states,
        audio_hidden_states,
        encoder_hidden_states,
        audio_encoder_hidden_states,
        timestep,
        audio_timestep,
        encoder_attention_mask,
        audio_encoder_attention_mask,
        sigma,
        audio_sigma,
    ):
        return self.transformer(
            hidden_states=hidden_states,
            audio_hidden_states=audio_hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            audio_encoder_hidden_states=audio_encoder_hidden_states,
            timestep=timestep,
            audio_timestep=audio_timestep,
            encoder_attention_mask=encoder_attention_mask,
            audio_encoder_attention_mask=audio_encoder_attention_mask,
            sigma=sigma,
            audio_sigma=audio_sigma,
            num_frames=2,
            height=2,
            width=2,
            audio_num_frames=4,
            return_dict=False,
        )

In [ ]:
text_wrapper = TextEncoderWrapper(pipe.text_encoder)
transformer_wrapper = TransformerWrapper(pipe.transformer)
vae_encoder_wrapper = VAEEncoderWrapper(pipe.vae)
vae_decoder_wrapper = VAEDecoderWrapper(pipe.vae)

text_inputs = (
    torch.ones((1, 16), dtype=torch.int64),
    torch.ones((1, 16), dtype=torch.int64),
)
transformer_inputs = (
    torch.zeros((1, 8, pipe.transformer.config.in_channels), dtype=torch.bfloat16),
    torch.zeros((1, 4, pipe.transformer.config.audio_in_channels), dtype=torch.bfloat16),
    torch.zeros((1, 16, pipe.transformer.config.cross_attention_dim), dtype=torch.bfloat16),
    torch.zeros((1, 16, pipe.transformer.config.audio_cross_attention_dim), dtype=torch.bfloat16),
    torch.zeros((1, 8), dtype=torch.float32),
    torch.zeros((1, 4), dtype=torch.float32),
    torch.ones((1, 16), dtype=torch.float32),
    torch.ones((1, 16), dtype=torch.float32),
    torch.zeros((1,), dtype=torch.float32),
    torch.zeros((1,), dtype=torch.float32),
)
vae_encoder_inputs = (torch.zeros((1, 3, 9, 64, 64), dtype=torch.float32),)
vae_decoder_inputs = (torch.zeros((1, pipe.vae.config.latent_channels, 2, 2, 2), dtype=torch.float32),)

In [ ]:
with torch.no_grad():
    ov_text = ov.convert_model(text_wrapper, example_input=text_inputs)
    ov_transformer = ov.convert_model(transformer_wrapper, example_input=transformer_inputs)
    ov_vae_encoder = ov.convert_model(vae_encoder_wrapper, example_input=vae_encoder_inputs)
    ov_vae_decoder = ov.convert_model(vae_decoder_wrapper, example_input=vae_decoder_inputs)

ov.save_model(ov_text, OUT_DIR / 'text_encoder.xml')
ov.save_model(ov_transformer, OUT_DIR / 'transformer.xml')
ov.save_model(ov_vae_encoder, OUT_DIR / 'vae_encoder.xml')
ov.save_model(ov_vae_decoder, OUT_DIR / 'vae_decoder.xml')

print('Saved OpenVINO models to', OUT_DIR.resolve())

## CPU / GPU smoke tests

In [ ]:
def compile_if_available(model_path, device):
    model = core.read_model(str(model_path))
    return core.compile_model(model, device)

available = core.available_devices
print('Available devices:', available)

compiled_cpu = {
    'text_encoder': compile_if_available(OUT_DIR / 'text_encoder.xml', 'CPU'),
    'transformer': compile_if_available(OUT_DIR / 'transformer.xml', 'CPU'),
    'vae_encoder': compile_if_available(OUT_DIR / 'vae_encoder.xml', 'CPU'),
    'vae_decoder': compile_if_available(OUT_DIR / 'vae_decoder.xml', 'CPU'),
}

compiled_gpu = {}
if any(device.startswith('GPU') for device in available):
    for name in ['text_encoder', 'transformer']:
        compiled_gpu[name] = compile_if_available(OUT_DIR / f'{name}.xml', 'GPU')

print('CPU compiled:', list(compiled_cpu))
print('GPU compiled:', list(compiled_gpu))

In [ ]:
prompt = ['A paper boat floating on a calm lake at sunset']
prompt_embeds, prompt_mask = pipe._get_gemma_prompt_embeds(prompt, max_sequence_length=16)
video_prompt_embeds, audio_prompt_embeds, connector_mask = pipe.connectors(prompt_embeds, prompt_mask)

transformer_request = {
    'hidden_states': np.zeros((1, 8, pipe.transformer.config.in_channels), dtype=np.float16),
    'audio_hidden_states': np.zeros((1, 4, pipe.transformer.config.audio_in_channels), dtype=np.float16),
    'encoder_hidden_states': video_prompt_embeds[:, :16].detach().cpu().to(torch.float16).numpy(),
    'audio_encoder_hidden_states': audio_prompt_embeds[:, :16].detach().cpu().to(torch.float16).numpy(),
    'timestep': np.zeros((1, 8), dtype=np.float32),
    'audio_timestep': np.zeros((1, 4), dtype=np.float32),
    'encoder_attention_mask': connector_mask[:, :16].detach().cpu().to(torch.float32).numpy(),
    'audio_encoder_attention_mask': connector_mask[:, :16].detach().cpu().to(torch.float32).numpy(),
    'sigma': np.zeros((1,), dtype=np.float32),
    'audio_sigma': np.zeros((1,), dtype=np.float32),
}

transformer_out = compiled_cpu['transformer'](transformer_request)
print('Transformer smoke test OK:', [tuple(value.shape) for value in transformer_out.values()])

## Reference video generation

This section produces a short reference clip with the original Diffusers pipeline. It is useful for visual validation while the OpenVINO components are still being integrated into a dedicated pipeline wrapper.

In [ ]:
generator = torch.Generator(device='cpu').manual_seed(42)
result = pipe(
    prompt='A paper boat floating on a calm lake at sunset',
    negative_prompt='blurry, distorted',
    num_frames=9,
    height=256,
    width=256,
    num_inference_steps=4,
    guidance_scale=3.0,
    generator=generator,
)

frames = result.frames[0]
out_video = Path('ltx23_reference.mp4')
iio.imwrite(out_video, frames, fps=8)
out_video

In [ ]:
from IPython.display import Video
Video('ltx23_reference.mp4', embed=True)